# 08. FT-Transformer 계열 모델 + LightGBM 앙상블

타 팀이 Transformer + XGBoost 앙상블로 점수를 올렸다는 얘기를 참고해서, 저희도 **정형 데이터용 Transformer(FT-Transformer 스타일: 피처를 토큰화 -> self-attention -> 분류)** 를 만들어 LightGBM(04, CV balanced accuracy 0.94987)과 앙상블했을 때 이득이 있는지 검증합니다.

**검증 절차**
1. Transformer를 04와 동일한 5-fold로 학습해서 OOF 확률 확보, 단독 성능(사전확률 보정 적용) 확인
2. LightGBM(04 파라미터)의 OOF 확률도 동일 폴드로 재생성
3. 두 OOF 확률을 가중평균(`alpha`)으로 블렌딩, `alpha`를 그리드서치해서 최적 조합 탐색
4. 블렌딩이 04 baseline(0.94987)보다 나은지 확인

**주의**: 저희 라벨은 `sleep_duration<6`, `>=7` 같은 날카로운 임계값 규칙으로 생성됨(핵심 발견 참고). Transformer 계열은 부드러운 결정경계를 만드는 경향이 있어 이런 hard threshold를 트리 모델만큼 잘 못 잡을 가능성이 있음 — 그래서 단독 성능은 LightGBM보다 낮을 것으로 예상하고, 앙상블 이득이 있는지가 핵심 질문.

커널: **Python (teammate)** (lightgbm 4.6.0 + torch 2.14, MPS 가속)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.metrics import balanced_accuracy_score, accuracy_score
import lightgbm as lgb
import torch
import torch.nn as nn

SEED = 42
N_FOLDS = 5
DATA_DIR = Path("../playground-series-s6e7")
OUT_DIR = DATA_DIR / "processed"
TARGET = "health_condition"

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
folds = pd.read_csv(OUT_DIR / "cv_folds.csv")
train = train.merge(folds, on="id", how="left")
assert train["fold"].isna().sum() == 0
print(train.shape, test.shape)

device: mps


(690088, 16) (295753, 14)


## 1. 결측 복구 피처 (04와 동일)

In [2]:
def add_recovery_features(df, step_tertiles, sleep_quality_cond):
    df = df.copy()
    activity_proxy = pd.cut(
        df["step_count"], bins=[-np.inf, step_tertiles[0], step_tertiles[1], np.inf],
        labels=["sedentary", "moderate", "active"],
    ).astype(object)
    df["physical_activity_level_recovered"] = df["physical_activity_level"]
    missing_activity = df["physical_activity_level"].isna()
    df.loc[missing_activity, "physical_activity_level_recovered"] = activity_proxy[missing_activity]
    df["physical_activity_level_recovered"] = df["physical_activity_level_recovered"].fillna("moderate")

    df["sleep_duration_recovered"] = df["sleep_duration"]
    missing_sleep = df["sleep_duration"].isna()
    fallback_median = sleep_quality_cond.get("missing", sleep_quality_cond["average"])
    mapped = df.loc[missing_sleep, "sleep_quality"].map(sleep_quality_cond).fillna(fallback_median)
    df.loc[missing_sleep, "sleep_duration_recovered"] = mapped

    df["stress_level_isnull"] = df["stress_level"].isna().astype(np.int8)
    df["sleep_duration_isnull"] = df["sleep_duration"].isna().astype(np.int8)
    df["physical_activity_level_isnull"] = df["physical_activity_level"].isna().astype(np.int8)
    return df


step_tertiles = train["step_count"].quantile([1/3, 2/3]).values
sleep_quality_cond = train.groupby("sleep_quality")["sleep_duration"].median().to_dict()
sleep_quality_cond["missing"] = train["sleep_duration"].median()

train = add_recovery_features(train, step_tertiles, sleep_quality_cond)
test = add_recovery_features(test, step_tertiles, sleep_quality_cond)

NUMERIC_COLS = [
    "sleep_duration_recovered", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
CATEGORICAL_COLS = [
    "stress_level", "sleep_quality", "physical_activity_level_recovered", "smoking_alcohol", "diet_type", "gender",
]
FLAG_COLS = ["stress_level_isnull", "sleep_duration_isnull", "physical_activity_level_isnull"]

## 2. Transformer 입력 준비

- 수치형: train 기준 median 대치 + StandardScaler 표준화
- 범주형: `"missing"`을 포함한 정수 인덱스로 인코딩 (임베딩 lookup용, LightGBM의 OrdinalEncoder와는 다른 목적)

In [3]:
numeric_medians = train[NUMERIC_COLS].median()
for df in (train, test):
    for col in NUMERIC_COLS:
        df[col] = df[col].fillna(numeric_medians[col])

scaler = StandardScaler()
train_numeric = scaler.fit_transform(train[NUMERIC_COLS]).astype(np.float32)
test_numeric = scaler.transform(test[NUMERIC_COLS]).astype(np.float32)

for df in (train, test):
    for col in CATEGORICAL_COLS:
        df[col] = df[col].fillna("missing")

cat_encoders = {}
cat_cardinalities = []
train_cat = np.zeros((len(train), len(CATEGORICAL_COLS)), dtype=np.int64)
test_cat = np.zeros((len(test), len(CATEGORICAL_COLS)), dtype=np.int64)
for i, col in enumerate(CATEGORICAL_COLS):
    enc = LabelEncoder()
    train_cat[:, i] = enc.fit_transform(train[col])
    # test에 새 카테고리가 없다는 전제(캐글 대회 특성상 카테고리 고정), 방어적으로 map 처리
    mapping = {c: j for j, c in enumerate(enc.classes_)}
    test_cat[:, i] = test[col].map(mapping).fillna(0).astype(np.int64)
    cat_encoders[col] = enc
    cat_cardinalities.append(len(enc.classes_))

train_flags = train[FLAG_COLS].values.astype(np.float32)
test_flags = test[FLAG_COLS].values.astype(np.float32)

target_encoder = LabelEncoder()
train_target = target_encoder.fit_transform(train[TARGET]).astype(np.int64)
class_order = list(target_encoder.classes_)
train_priors = train[TARGET].value_counts(normalize=True).to_dict()

print("numeric:", train_numeric.shape, "categorical cardinalities:", cat_cardinalities, "flags:", train_flags.shape)

numeric: (690088, 7) categorical cardinalities: [4, 4, 3, 4, 4, 4] flags: (690088, 3)


## 3. FT-Transformer-lite 모델 정의

- 수치형 피처마다 학습되는 선형 토크나이저(값 -> d_model 벡터)
- 범주형 피처마다 임베딩 테이블
- 결측 플래그는 별도 수치 토큰으로 추가
- 전부 토큰 시퀀스로 만들고 [CLS] 토큰 추가 -> TransformerEncoder -> [CLS] 출력을 분류 head에 통과

In [4]:
class FTTransformerLite(nn.Module):
    def __init__(self, n_numeric, n_flags, cat_cardinalities, d_model=32, n_heads=4, n_layers=2, n_classes=3, dropout=0.1):
        super().__init__()
        self.n_numeric = n_numeric
        self.n_flags = n_flags
        n_num_tokens = n_numeric + n_flags

        # 수치형(+ 플래그) 피처별 선형 토크나이저: 각 피처마다 독립적인 (weight, bias)
        self.numeric_weight = nn.Parameter(torch.randn(n_num_tokens, d_model) * 0.02)
        self.numeric_bias = nn.Parameter(torch.zeros(n_num_tokens, d_model))

        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(card, d_model) for card in cat_cardinalities
        ])

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model, n_classes),
        )

    def forward(self, x_num, x_cat):
        # x_num: (B, n_num_tokens), x_cat: (B, n_cat)
        num_tokens = x_num.unsqueeze(-1) * self.numeric_weight + self.numeric_bias  # (B, n_num_tokens, d_model)
        cat_tokens = torch.stack(
            [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeddings)], dim=1
        )  # (B, n_cat, d_model)
        cls = self.cls_token.expand(x_num.size(0), -1, -1)
        tokens = torch.cat([cls, num_tokens, cat_tokens], dim=1)
        encoded = self.encoder(tokens)
        cls_out = encoded[:, 0]
        return self.head(cls_out)


print("model class ready")

model class ready


## 4. 5-fold 학습 (OOF 확률 수집)

In [5]:
def prior_corrected_predict(proba, class_order, priors):
    prior_arr = np.array([priors[c] for c in class_order])
    scores = proba / prior_arr
    return np.array(class_order)[scores.argmax(axis=1)]


def train_transformer_fold(tr_idx, va_idx, x_num, x_flags, x_cat, y, epochs=15, batch_size=4096, lr=1e-3):
    model = FTTransformerLite(
        n_numeric=x_num.shape[1], n_flags=x_flags.shape[1], cat_cardinalities=cat_cardinalities,
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()

    x_num_full = np.concatenate([x_num, x_flags], axis=1)
    tr_num = torch.tensor(x_num_full[tr_idx], device=device)
    tr_cat = torch.tensor(x_cat[tr_idx], device=device)
    tr_y = torch.tensor(y[tr_idx], device=device)
    va_num = torch.tensor(x_num_full[va_idx], device=device)
    va_cat = torch.tensor(x_cat[va_idx], device=device)
    va_y = torch.tensor(y[va_idx], device=device)

    n_train = tr_num.size(0)
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n_train, device=device)
        total_loss = 0.0
        for start in range(0, n_train, batch_size):
            idx = perm[start:start + batch_size]
            opt.zero_grad()
            logits = model(tr_num[idx], tr_cat[idx])
            loss = loss_fn(logits, tr_y[idx])
            loss.backward()
            opt.step()
            total_loss += loss.item() * idx.size(0)
        sched.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(va_num, va_cat)
            val_loss = loss_fn(val_logits, va_y).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"  epoch {epoch+1}/{epochs} train_loss={total_loss/n_train:.4f} val_loss={val_loss:.4f}")

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_proba = torch.softmax(model(va_num, va_cat), dim=1).cpu().numpy()
    return model, val_proba


oof_proba_transformer = np.zeros((len(train), 3))
fold_models = []
fold_arr = train["fold"].values

for fold in range(N_FOLDS):
    print(f"=== fold {fold} ===")
    tr_idx = np.where(fold_arr != fold)[0]
    va_idx = np.where(fold_arr == fold)[0]
    model, val_proba = train_transformer_fold(tr_idx, va_idx, train_numeric, train_flags, train_cat, train_target)
    oof_proba_transformer[va_idx] = val_proba
    fold_models.append(model)

pred_transformer = prior_corrected_predict(oof_proba_transformer, class_order, train_priors)
ba_transformer = balanced_accuracy_score(train[TARGET].values, pred_transformer)
acc_transformer = accuracy_score(train[TARGET].values, pred_transformer)
print(f"\nTransformer 단독 CV balanced accuracy: {ba_transformer:.5f} (accuracy {acc_transformer:.5f})")

=== fold 0 ===


  epoch 1/15 train_loss=0.3491 val_loss=0.1290


  epoch 2/15 train_loss=0.1261 val_loss=0.1028


  epoch 3/15 train_loss=0.1113 val_loss=0.0991


  epoch 4/15 train_loss=0.1065 val_loss=0.0960


...

  epoch 12/15 train_loss=0.0979 val_loss=0.0965


  epoch 13/15 train_loss=0.0978 val_loss=0.0966


  epoch 14/15 train_loss=0.0977 val_loss=0.0964


  epoch 15/15 train_loss=0.0976 val_loss=0.0964



Transformer 단독 CV balanced accuracy: 0.94753 (accuracy 0.93486)


## 5. LightGBM(04) OOF 확률 재생성

In [6]:
ORDINAL_COLS = {
    "stress_level": ["low", "medium", "high"],
    "sleep_quality": ["poor", "average", "good"],
    "physical_activity_level_recovered": ["sedentary", "moderate", "active"],
    "smoking_alcohol": ["no", "occasional", "yes"],
}
NOMINAL_COLS = ["diet_type", "gender"]

lgb_train = train.copy()
lgb_test = test.copy()
for col in list(ORDINAL_COLS.keys()) + NOMINAL_COLS:
    lgb_train[col] = lgb_train[col].fillna("missing")
    lgb_test[col] = lgb_test[col].fillna("missing")

for col, order in ORDINAL_COLS.items():
    categories = order + ["missing"]
    enc = OrdinalEncoder(categories=[categories])
    lgb_train[col] = enc.fit_transform(lgb_train[[col]])
    lgb_test[col] = enc.transform(lgb_test[[col]])

lgb_train_ohe = pd.get_dummies(lgb_train[NOMINAL_COLS], prefix=NOMINAL_COLS)
lgb_test_ohe = pd.get_dummies(lgb_test[NOMINAL_COLS], prefix=NOMINAL_COLS).reindex(columns=lgb_train_ohe.columns, fill_value=0)
lgb_train = pd.concat([lgb_train.drop(columns=NOMINAL_COLS), lgb_train_ohe], axis=1)
lgb_test = pd.concat([lgb_test.drop(columns=NOMINAL_COLS), lgb_test_ohe], axis=1)

LGB_NUMERIC = ["sleep_duration_recovered", "heart_rate", "bmi", "calorie_expenditure", "step_count", "exercise_duration", "water_intake"]
lgb_numeric_medians = lgb_train[LGB_NUMERIC].median()
for df in (lgb_train, lgb_test):
    for col in LGB_NUMERIC:
        df[col] = df[col].fillna(lgb_numeric_medians[col])

LGB_FEATURE_COLS = LGB_NUMERIC + list(ORDINAL_COLS.keys()) + FLAG_COLS + list(lgb_train_ohe.columns)
lgb_train["target_enc"] = target_encoder.transform(lgb_train[TARGET])

tuned_params = dict(
    objective="multiclass", num_class=3, random_state=SEED, verbosity=-1,
    n_estimators=500,
    learning_rate=0.047792122422826176,
    num_leaves=19,
    max_depth=9,
    min_child_samples=172,
    subsample=0.8929201812783885,
    colsample_bytree=0.7318659835628288,
    reg_alpha=0.0005023614837892232,
    reg_lambda=0.32275087452118445,
)

oof_proba_lgb = np.zeros((len(lgb_train), 3))
for fold in range(N_FOLDS):
    tr_idx = lgb_train["fold"] != fold
    va_idx = lgb_train["fold"] == fold
    X_tr, y_tr = lgb_train.loc[tr_idx, LGB_FEATURE_COLS], lgb_train.loc[tr_idx, "target_enc"]
    X_va, y_va = lgb_train.loc[va_idx, LGB_FEATURE_COLS], lgb_train.loc[va_idx, "target_enc"]
    model = lgb.LGBMClassifier(**tuned_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_proba_lgb[va_idx.values] = model.predict_proba(X_va)
    print(f"lgb fold {fold} done")

pred_lgb = prior_corrected_predict(oof_proba_lgb, class_order, train_priors)
ba_lgb = balanced_accuracy_score(train[TARGET].values, pred_lgb)
print(f"\nLightGBM 단독 CV balanced accuracy (재현): {ba_lgb:.5f} (04 원본: 0.94987)")

lgb fold 0 done


lgb fold 1 done


lgb fold 2 done


lgb fold 3 done


lgb fold 4 done



LightGBM 단독 CV balanced accuracy (재현): 0.94988 (04 원본: 0.94987)


## 6. 블렌딩 가중치 탐색

In [7]:
blend_results = []
for alpha in np.arange(0.0, 1.01, 0.05):
    blended = alpha * oof_proba_lgb + (1 - alpha) * oof_proba_transformer
    pred = prior_corrected_predict(blended, class_order, train_priors)
    ba = balanced_accuracy_score(train[TARGET].values, pred)
    blend_results.append({"alpha_lgb": round(alpha, 2), "balanced_accuracy": ba})

blend_df = pd.DataFrame(blend_results)
best_blend = blend_df.loc[blend_df["balanced_accuracy"].idxmax()]
print(blend_df.to_string(index=False))
print(f"\n최적 alpha(LightGBM 가중치): {best_blend['alpha_lgb']}, BA: {best_blend['balanced_accuracy']:.5f}")
print(f"LightGBM 단독: {ba_lgb:.5f} | Transformer 단독: {ba_transformer:.5f} | 최적 블렌딩: {best_blend['balanced_accuracy']:.5f}")
print(f"04 baseline 대비 개선폭: {best_blend['balanced_accuracy'] - 0.94987:+.5f}")

 alpha_lgb  balanced_accuracy
      0.00           0.947534
      0.05           0.947706
      0.10           0.947815
      0.15           0.947984
      0.20           0.948053
      0.25           0.948170
      0.30           0.948249
      0.35           0.948360
      0.40           0.948467
      0.45           0.948601
      0.50           0.948716
      0.55           0.948870
      0.60           0.949011
      0.65           0.949197
      0.70           0.949413
      0.75           0.949583
      0.80           0.949766
      0.85           0.949770
      0.90           0.949867
      0.95           0.949943
      1.00           0.949880

최적 alpha(LightGBM 가중치): 0.95, BA: 0.94994
LightGBM 단독: 0.94988 | Transformer 단독: 0.94753 | 최적 블렌딩: 0.94994
04 baseline 대비 개선폭: +0.00007


## 7. 개선 확인되면 최종 제출 파일 생성

In [8]:
if best_blend["balanced_accuracy"] > 0.94987:
    print("개선 확인 -> 전체 데이터로 최종 모델 재학습 후 test 예측 생성")

    final_lgb = lgb.LGBMClassifier(**tuned_params)
    final_lgb.fit(lgb_train[LGB_FEATURE_COLS], lgb_train["target_enc"])
    test_proba_lgb = final_lgb.predict_proba(lgb_test[LGB_FEATURE_COLS])

    # transformer: 5-fold 모델들의 test 예측을 평균 (앙상블)
    test_num_full = np.concatenate([test_numeric, test_flags], axis=1)
    test_num_t = torch.tensor(test_num_full, device=device)
    test_cat_t = torch.tensor(test_cat, device=device)
    test_proba_transformer = np.zeros((len(test), 3))
    with torch.no_grad():
        for m in fold_models:
            m.eval()
            test_proba_transformer += torch.softmax(m(test_num_t, test_cat_t), dim=1).cpu().numpy()
    test_proba_transformer /= len(fold_models)

    alpha = best_blend["alpha_lgb"]
    test_proba_blend = alpha * test_proba_lgb + (1 - alpha) * test_proba_transformer
    test_pred = prior_corrected_predict(test_proba_blend, class_order, train_priors)

    submission = pd.DataFrame({"id": test["id"], TARGET: test_pred})
    submission.to_csv(OUT_DIR / "submission_v6_transformer_blend.csv", index=False)
    print("저장:", OUT_DIR / "submission_v6_transformer_blend.csv")
    print(submission[TARGET].value_counts(normalize=True))
else:
    print("04 대비 개선 없음 -> submission_v2_tuned.csv를 그대로 유지")

개선 확인 -> 전체 데이터로 최종 모델 재학습 후 test 예측 생성


저장: ../playground-series-s6e7/processed/submission_v6_transformer_blend.csv
health_condition
at-risk      0.809909
unhealthy    0.116093
fit          0.073998
Name: proportion, dtype: float64
